# Reusable narwhals pipelines with `ccflow`

This notebook walks through three layers of composition for building data-frame pipelines
on top of [narwhals](https://narwhals-dev.github.io/narwhals/), provided by `ccflow.models.narwhals`:

1. **`NarwhalsFrameTransform`** — a pure `LazyFrame -> LazyFrame` step. Framework-agnostic;
   usable standalone via `lf.pipe(transform)` without any other ccflow machinery.
2. **`SequenceTransform`** — bundles a list of transforms into a single transform.
3. **`NarwhalsPipelineModel`** — a `CallableModel` that wires a frame source to a list of
   transforms, producing a new `NarwhalsFrameResult`.

Plus two generic implementations:

- **`JoinTransform`** — joins another callable model's frame onto the input frame.
- **`JoinBackTransform`** — runs an inner transform on the input, joins the result back.

The TPC-H benchmark is used as a worked example (Q1 → Q3).

## Why?

A typical narwhals query is just a function:

```python
def query(lineitem):
    return (
        lineitem.filter(...)
        .with_columns(...)
        .group_by(...).agg(...)
        .sort(...)
    )
```

This is fine for a one-shot script. But in a real codebase you usually want:

- **Configurability** — the date cutoff, the segment, the group-by columns shouldn't be hard-coded.
- **Reusability** — small pieces (e.g. "filter by ship date") used in many queries.
- **Dependency injection** — swap the data source (DuckDB, parquet, mocked) without rewriting the query.
- **Serialization** — store the entire pipeline as JSON or YAML; reload from config.

The three layers below give you these properties incrementally — pay only for the level
of structure you actually need.

## The three layers, at a glance

| Layer | What it is | What you get |
|-------|------------|--------------|
| `NarwhalsFrameTransform` | A pydantic model that is callable on a `LazyFrame`. | Configurable, JSON-serializable, `.pipe()`-compatible. **No ccflow runtime required.** |
| `SequenceTransform` | A transform whose state is a list of transforms. | Composition + JSON roundtrip + nestable. Still just `.pipe()`-compatible. |
| `NarwhalsPipelineModel` | A `CallableModel` returning `NarwhalsFrameResult`. | Sources, contexts, dependency graph, evaluators, caching — full ccflow integration. |

You can stop at layer 1 if you only want better-behaved transforms in an existing codebase.
You can stop at layer 2 if you want bundling without ccflow.
You graduate to layer 3 when you need sourcing, configuration, or graph composition.

## 1. Setup & the starting point

We use the [TPC-H benchmark](https://www.tpc.org/tpch/) as our working data — a synthetic
sales/orders schema with 8 tables. ccflow ships a registry-driven TPC-H example
([`ccflow/examples/tpch/`](../examples/tpch/)) that exposes one `CallableModel` per table
under `/table/<name>` and one per query under `/query/Q<N>`. We load it once at scale
factor `0.01` (~10 MB per table) and pull tables straight off the registry.

In [1]:
import tempfile
from datetime import datetime
from pathlib import Path

import narwhals.stable.v1 as nw
import polars as pl
from polars.testing import assert_frame_equal

from ccflow import (
    BaseModel,
    CallableModel,
    Flow,
    JoinBackTransform,
    JoinTransform,
    ModelRegistry,
    NarwhalsFrameTransform,
    NarwhalsPipelineModel,
    NullContext,
    SequenceTransform,
)
from ccflow.examples.tpch import load_config
from ccflow.result.narwhals import NarwhalsFrameResult

# Populate the root registry from the bundled TPC-H config at SF=0.01.
load_config(overrides=["tpch.backend.scale_factor=0.01"])
registry = ModelRegistry.root()

### Q1 today

Here is TPC-H Q1, written as a plain narwhals function — adapted from
[`ccflow/examples/tpch/queries/q1.py`](../examples/tpch/queries/q1.py). It computes
pricing summary statistics for line items shipped before a cutoff date, grouped by
return-flag and line-status.

In [2]:
def q1_monolith(lineitem):
    cutoff = datetime(1998, 9, 2)
    return (
        lineitem.filter(nw.col("l_shipdate") <= cutoff)
        .with_columns(
            disc_price=nw.col("l_extendedprice") * (1 - nw.col("l_discount")),
            charge=nw.col("l_extendedprice") * (1 - nw.col("l_discount")) * (1 + nw.col("l_tax")),
        )
        .group_by("l_returnflag", "l_linestatus")
        .agg(
            nw.sum("l_quantity").alias("sum_qty"),
            nw.sum("l_extendedprice").alias("sum_base_price"),
            nw.sum("disc_price").alias("sum_disc_price"),
            nw.sum("charge").alias("sum_charge"),
            nw.mean("l_quantity").alias("avg_qty"),
            nw.mean("l_extendedprice").alias("avg_price"),
            nw.mean("l_discount").alias("avg_disc"),
            nw.len().alias("count_order"),
        )
        .sort("l_returnflag", "l_linestatus")
    )


# Pull the lineitem table from the registry. Each table provider is a NullContext-typed
# CallableModel, so calling it with no args returns an eager NarwhalsDataFrameResult.
lineitem_eager = registry["/table/lineitem"]().df
lineitem = lineitem_eager.lazy()  # work lazily for plan optimization

q1_monolith(lineitem).collect().to_native()

l_returnflag,l_linestatus,sum_qty,sum_base_price,sum_disc_price,sum_charge,avg_qty,avg_price,avg_disc,count_order
str,str,f64,f64,f64,f64,f64,f64,f64,u32
"""A""","""F""",380456.0,5.3235e8,5.0582e8,5.2617e8,25.575155,35785.709307,0.050081,14876
"""N""","""F""",8971.0,1.2385e7,1.1798e7,1.2282e7,25.778736,35588.509684,0.047759,348
"""N""","""O""",742802.0,1.0415e9,9.8974e8,1.0294e9,25.454988,35691.129209,0.049931,29181
"""R""","""F""",381449.0,5.3459e8,5.0800e8,5.2852e8,25.597168,35874.006533,0.049828,14902


### What's wrong with this picture?

This works, but the function locks in:

- A specific **date cutoff** (`1998-09-02`).
- A specific **set of group-by columns**.
- A specific **set of aggregation outputs**.
- The expectation that the input is exactly `lineitem` from a hard-coded place.

Each of those is a reasonable thing to *configure*. And while we can certainly add parameters
and turn this into a 12-arg function, that scales poorly: it conflates "what the pipeline does"
with "where the data comes from", and gives no good way to swap individual stages.

The next sections build up structure gradually until each of these concerns has its own home.

## 2. The `NarwhalsFrameTransform` base class

A `NarwhalsFrameTransform` is just a callable pydantic model that takes a `narwhals.LazyFrame`
and returns a `narwhals.LazyFrame`. Subclasses declare configuration via fields and implement
`__call__`.

Crucially: **a transform is usable on its own, with no ccflow runtime required.** It works
directly as the argument to `narwhals.LazyFrame.pipe(...)`.

In [3]:
class MultiplyColumn(NarwhalsFrameTransform):
    """Toy transform: multiply a column by a constant factor."""

    col: str
    factor: float

    def __call__(self, df):
        return df.with_columns(nw.col(self.col) * self.factor)


lf = nw.from_native(pl.LazyFrame({"x": [1, 2, 3]}))

t = MultiplyColumn(col="x", factor=10.0)
print("config:", t.model_dump_json())
print("output:", lf.pipe(t).collect().to_native())

config: {"col":"x","factor":10.0,"type_":"ccflow.local_persistence._Local_MultiplyColumn_2bb470e88fda"}
output: shape: (3, 1)
┌──────┐
│ x    │
│ ---  │
│ f64  │
╞══════╡
│ 10.0 │
│ 20.0 │
│ 30.0 │
└──────┘


That's it. `MultiplyColumn` reuses cleanly anywhere narwhals is used — no ccflow imports,
sources, or contexts are needed in client code that just wants to call `lf.pipe(t)`.

The benefits of being a pydantic model show up when you start composing or persisting:
field validation, JSON roundtrip, IDE autocompletion, hydra config support.

## 3. Refactoring Q1 into business-meaningful transforms

The Q1 monolith naturally decomposes into four phases. Each becomes its own
`NarwhalsFrameTransform`, each named after what it *does* in business terms.

### 3.1. Filter by ship date

In [4]:
class FilterByShipDate(NarwhalsFrameTransform):
    cutoff: datetime

    def __call__(self, df):
        return df.filter(nw.col("l_shipdate") <= self.cutoff)


filtered = lineitem.pipe(FilterByShipDate(cutoff=datetime(1998, 9, 2)))
filtered.collect().shape

(59307, 16)

### 3.2. Derive line-item metrics

In [5]:
class DeriveLineItemMetrics(NarwhalsFrameTransform):
    """Adds disc_price = extendedprice * (1 - discount), charge = disc_price * (1 + tax)."""

    def __call__(self, df):
        return df.with_columns(
            disc_price=nw.col("l_extendedprice") * (1 - nw.col("l_discount")),
            charge=nw.col("l_extendedprice") * (1 - nw.col("l_discount")) * (1 + nw.col("l_tax")),
        )


lineitem.pipe(FilterByShipDate(cutoff=datetime(1998, 9, 2))).pipe(DeriveLineItemMetrics()).collect().columns

['l_orderkey',
 'l_partkey',
 'l_suppkey',
 'l_linenumber',
 'l_quantity',
 'l_extendedprice',
 'l_discount',
 'l_tax',
 'l_returnflag',
 'l_linestatus',
 'l_shipdate',
 'l_commitdate',
 'l_receiptdate',
 'l_shipinstruct',
 'l_shipmode',
 'l_comment',
 'disc_price',
 'charge']

### 3.3. Summarize by return-status

Aggregate by `(l_returnflag, l_linestatus)` and sort the result deterministically. The sort
keys are the same as the group keys, so it's natural to bundle them into one transform.

In [6]:
class SummarizeByReturnStatus(NarwhalsFrameTransform):
    def __call__(self, df):
        return (
            df.group_by("l_returnflag", "l_linestatus")
            .agg(
                nw.sum("l_quantity").alias("sum_qty"),
                nw.sum("l_extendedprice").alias("sum_base_price"),
                nw.sum("disc_price").alias("sum_disc_price"),
                nw.sum("charge").alias("sum_charge"),
                nw.mean("l_quantity").alias("avg_qty"),
                nw.mean("l_extendedprice").alias("avg_price"),
                nw.mean("l_discount").alias("avg_disc"),
                nw.len().alias("count_order"),
            )
            .sort("l_returnflag", "l_linestatus")
        )

### Compose by hand

Now we can assemble the query as a chain of `.pipe()` calls. The chain reads top-to-bottom
like a description of what the query *does*.

In [7]:
result = lineitem.pipe(FilterByShipDate(cutoff=datetime(1998, 9, 2))).pipe(DeriveLineItemMetrics()).pipe(SummarizeByReturnStatus())
result.collect().to_native()

l_returnflag,l_linestatus,sum_qty,sum_base_price,sum_disc_price,sum_charge,avg_qty,avg_price,avg_disc,count_order
str,str,f64,f64,f64,f64,f64,f64,f64,u32
"""A""","""F""",380456.0,5.3235e8,5.0582e8,5.2617e8,25.575155,35785.709307,0.050081,14876
"""N""","""F""",8971.0,1.2385e7,1.1798e7,1.2282e7,25.778736,35588.509684,0.047759,348
"""N""","""O""",742802.0,1.0415e9,9.8974e8,1.0294e9,25.454988,35691.129209,0.049931,29181
"""R""","""F""",381449.0,5.3459e8,5.0800e8,5.2852e8,25.597168,35874.006533,0.049828,14902


## 4. Aside: do you *have* to subclass `NarwhalsFrameTransform`?

**No.** `NarwhalsFrameTransform` is a recommended convention, not a requirement. The
pipeline machinery accepts three flavours of "transform":

| Flavour                      | `.pipe()` works? | Configurable params? | JSON round-trip? | Notes |
|------------------------------|:----------------:|:--------------------:|:----------------:|-------|
| Plain function / lambda      | ✅               | (closure only)       | ❌               | Quickest for one-offs and pipeline-local logic. |
| Plain `ccflow.BaseModel`     | ✅               | ✅                   | ✅               | Round-trips via ccflow's `type_` discriminator. |
| `NarwhalsFrameTransform`     | ✅               | ✅                   | ✅               | Adds a typed contract, codebase searchability, and a hook point for future cross-cutting features (validation, tracing). |

The `transforms` field on both `SequenceTransform` and `NarwhalsPipelineModel` is typed as
`Union[NarwhalsFrameTransform, Callable]`. The `NarwhalsFrameTransform` branch is what
enables `type_`-based deserialization for any model-shaped transform; the `Callable` branch
is the ergonomic escape hatch for plain functions (which are accepted at runtime but cannot
be JSON serialized).

**Rule of thumb:** reach for `NarwhalsFrameTransform` when the transform is reusable across
pipelines or you want it to be discoverable in your codebase. Use a plain function for
ad-hoc, pipeline-local logic.

In [8]:
class PlainModelDoubler(BaseModel):
    """A plain ccflow.BaseModel, not subclassing NarwhalsFrameTransform."""

    col: str

    def __call__(self, df):
        return df.with_columns(nw.col(self.col) * 2)


def add_constant(df, c=10):
    """A plain function; bind args via functools.partial or a lambda."""
    return df.with_columns(nw.col("l_quantity") + c)


demo = SequenceTransform(
    transforms=[
        PlainModelDoubler(col="l_quantity"),  # plain BaseModel
        lambda df: add_constant(df, c=5),  # lambda
        DeriveLineItemMetrics(),  # NarwhalsFrameTransform
    ]
)
lineitem.pipe(demo).select("l_quantity", "disc_price").head().collect().to_native()

l_quantity,disc_price
f64,f64
39.0,23721.936
77.0,51586.1892
21.0,11070.936
61.0,23493.0696
53.0,24650.784


## 5. `SequenceTransform`: bundling transforms

When you find yourself repeating the same chain of transforms in multiple places, bundle
them into a `SequenceTransform`. The bundle is itself a `NarwhalsFrameTransform`, so it
remains `.pipe()`-compatible and can be nested inside other sequences.

`SequenceTransform.transforms` accepts the same union as `NarwhalsPipelineModel.transforms`
(see §4): model-shaped transforms round-trip through JSON; plain callables work but don't
serialize.

In [9]:
q1_preproc = SequenceTransform(
    transforms=[
        FilterByShipDate(cutoff=datetime(1998, 9, 2)),
        DeriveLineItemMetrics(),
        SummarizeByReturnStatus(),
    ]
)

lineitem.pipe(q1_preproc).collect().head().to_native()

l_returnflag,l_linestatus,sum_qty,sum_base_price,sum_disc_price,sum_charge,avg_qty,avg_price,avg_disc,count_order
str,str,f64,f64,f64,f64,f64,f64,f64,u32
"""A""","""F""",380456.0,5.3235e8,5.0582e8,5.2617e8,25.575155,35785.709307,0.050081,14876
"""N""","""F""",8971.0,1.2385e7,1.1798e7,1.2282e7,25.778736,35588.509684,0.047759,348
"""N""","""O""",742802.0,1.0415e9,9.8974e8,1.0294e9,25.454988,35691.129209,0.049931,29181
"""R""","""F""",381449.0,5.3459e8,5.0800e8,5.2852e8,25.597168,35874.006533,0.049828,14902


In [10]:
serialized = q1_preproc.model_dump_json(indent=2)
print(serialized[:500] + " ...")

q1_preproc_restored = SequenceTransform.model_validate_json(serialized)
restored = lineitem.pipe(q1_preproc_restored).collect().to_native()
original = lineitem.pipe(q1_preproc).collect().to_native()
assert_frame_equal(restored, original, check_exact=False)
print("\nroundtrip OK ✓")

{
  "transforms": [
    {
      "cutoff": "1998-09-02T00:00:00",
      "type_": "ccflow.local_persistence._Local_FilterByShipDate_cbda01c65885"
    },
    {
      "type_": "ccflow.local_persistence._Local_DeriveLineItemMetrics_5bee7b397e22"
    },
    {
      "type_": "ccflow.local_persistence._Local_SummarizeByReturnStatus_01a29c30aedb"
    }
  ],
  "type_": "ccflow.models.narwhals.SequenceTransform"
} ...

roundtrip OK ✓


## 6. `NarwhalsPipelineModel`: source + transforms

`SequenceTransform` is still backend-agnostic — it just composes transforms. To get
**dependency injection**, **graph composition**, and **full configuration via Hydra/JSON**,
graduate to `NarwhalsPipelineModel`.

A pipeline takes:

- a **`source`**: any `CallableModel` whose `result_type` is `NarwhalsFrameResult` (or a subclass).
- a list of **`transforms`**: applied in order via `.pipe()`.

The pipeline is itself a `CallableModel` returning `NarwhalsFrameResult`. Its
`context_type` is delegated to the source — so the pipeline transparently adopts whatever
context type the source uses, and forwards the caller-supplied context to the source at
call time.

The output is **always lazy**: even if a transform returns an eager frame, the pipeline
re-coerces to a `LazyFrame` so subsequent stages can rely on the lazy contract. Users that
want an eager result call `.collect()` on the returned `NarwhalsFrameResult`.

### A source for the lineitem table

The TPC-H registry already exposes a `NullContext`-typed `TPCHTableProvider` for each
table at `/table/<name>`, so we can drop one straight in as the pipeline source.

In [11]:
q1_pipeline = NarwhalsPipelineModel(
    source=registry["/table/lineitem"],
    transforms=[
        FilterByShipDate(cutoff=datetime(1998, 9, 2)),
        DeriveLineItemMetrics(),
        SummarizeByReturnStatus(),
    ],
)

# The pipeline takes the source's context type directly.
print("pipeline context_type:", q1_pipeline.context_type.__name__)

q1_result = q1_pipeline()
print("type:", type(q1_result).__name__)
print("frame is lazy:", isinstance(q1_result.df, nw.LazyFrame))
q1_result.df.collect().to_native()

pipeline context_type: ContextBase
type: NarwhalsFrameResult
frame is lazy: True


l_returnflag,l_linestatus,sum_qty,sum_base_price,sum_disc_price,sum_charge,avg_qty,avg_price,avg_disc,count_order
str,str,f64,f64,f64,f64,f64,f64,f64,u32
"""A""","""F""",380456.0,5.3235e8,5.0582e8,5.2617e8,25.575155,35785.709307,0.050081,14876
"""N""","""F""",8971.0,1.2385e7,1.1798e7,1.2282e7,25.778736,35588.509684,0.047759,348
"""N""","""O""",742802.0,1.0415e9,9.8974e8,1.0294e9,25.454988,35691.129209,0.049931,29181
"""R""","""F""",381449.0,5.3459e8,5.0800e8,5.2852e8,25.597168,35874.006533,0.049828,14902


### Dependency injection: swap the source

The same transform list works against any source returning a `NarwhalsFrameResult`. To
demonstrate, here's a parquet-backed source that round-trips lineitem through disk.

In [12]:
class ParquetSource(CallableModel):
    """A NullContext-shaped source that reads a parquet file lazily."""

    path: Path

    @Flow.call
    def __call__(self, context: NullContext = NullContext()) -> NarwhalsFrameResult:
        return NarwhalsFrameResult(df=pl.scan_parquet(str(self.path)))


tmpdir = Path(tempfile.mkdtemp())
parquet_path = tmpdir / "lineitem.parquet"
lineitem_eager.to_native().write_parquet(parquet_path)

q1_from_parquet = NarwhalsPipelineModel(
    source=ParquetSource(path=parquet_path),
    transforms=q1_pipeline.transforms,  # exact same transforms, different source
)

assert_frame_equal(
    q1_from_parquet().df.collect().to_native(),
    q1_pipeline().df.collect().to_native(),
    check_exact=False,
)
print("DI swap OK ✓")

DI swap OK ✓


### Full pipeline serialization

Because every component is a pydantic model, the entire pipeline serializes as JSON or YAML
and reloads via Hydra. Here's a JSON snapshot of just the structure:

In [13]:
print(q1_pipeline.model_dump_json(indent=2)[:600], "\n  ...")

{
  "meta": {
    "name": "",
    "description": "",
    "type_": "ccflow.callable.MetaData"
  },
  "source": {
    "meta": {
      "name": "lineitem",
      "description": "",
      "type_": "ccflow.callable.MetaData"
    },
    "backend": {
      "scale_factor": 0.01,
      "type_": "ccflow.examples.tpch.data_generators.TPCHDuckDBBackend"
    },
    "table": "lineitem",
    "type_": "ccflow.examples.tpch.data_generators.TPCHTableProvider"
  },
  "transforms": [
    {
      "cutoff": "1998-09-02T00:00:00",
      "type_": "ccflow.local_persistence._Local_FilterByShipDate_cbda01c65885"
    },
  
  ...


The same data renders directly to YAML. A typical production setup would have a config
that references registry entries:

```yaml
_target_: ccflow.NarwhalsPipelineModel
source: /table/lineitem        # cross-reference to the registry-provided table
transforms:
  - _target_: my_project.FilterByShipDate
    cutoff: 1998-09-02
  - _target_: my_project.DeriveLineItemMetrics
  - _target_: my_project.SummarizeByReturnStatus
```

(See ccflow's Hydra integration docs and `ccflow/examples/tpch/config/conf.yaml` for the
full config-file workflow with cross-references.)

## 7. Multi-source enrichment with `JoinTransform`

Most queries involve more than one table. `JoinTransform` is a transform that takes the
input frame and joins another `CallableModel`'s frame onto it.

The signature mirrors `LazyFrame.join`. Both same-named (`on=...`) and cross-named
(`left_on=..., right_on=...`) join keys are supported.

### TPC-H Q3

Q3 picks the top 10 customer-segment shipments unshipped before a given date and groups
them by order. Three tables: customer, orders, lineitem.

In [14]:
class FilterCustomerSegment(NarwhalsFrameTransform):
    segment: str

    def __call__(self, df):
        return df.filter(nw.col("c_mktsegment") == self.segment)


class FilterOrderDateBefore(NarwhalsFrameTransform):
    cutoff: datetime

    def __call__(self, df):
        return df.filter(nw.col("o_orderdate") < self.cutoff)


class FilterShipDateAfter(NarwhalsFrameTransform):
    cutoff: datetime

    def __call__(self, df):
        return df.filter(nw.col("l_shipdate") > self.cutoff)


class DeriveRevenue(NarwhalsFrameTransform):
    def __call__(self, df):
        return df.with_columns((nw.col("l_extendedprice") * (1 - nw.col("l_discount"))).alias("revenue"))


class AggregateOrderRevenue(NarwhalsFrameTransform):
    def __call__(self, df):
        return (
            df.group_by(["o_orderkey", "o_orderdate", "o_shippriority"])
            .agg(nw.sum("revenue"))
            .select(
                nw.col("o_orderkey").alias("l_orderkey"),
                "revenue",
                "o_orderdate",
                "o_shippriority",
            )
            .sort(by=["revenue", "o_orderdate"], descending=[True, False])
            .head(10)
        )


cutoff = datetime(1995, 3, 15)

q3_pipeline = NarwhalsPipelineModel(
    source=registry["/table/customer"],
    transforms=[
        FilterCustomerSegment(segment="BUILDING"),
        JoinTransform(
            other=registry["/table/orders"],
            left_on="c_custkey",
            right_on="o_custkey",
            how="inner",
        ),
        JoinTransform(
            other=registry["/table/lineitem"],
            left_on="o_orderkey",
            right_on="l_orderkey",
            how="inner",
        ),
        FilterOrderDateBefore(cutoff=cutoff),
        FilterShipDateAfter(cutoff=cutoff),
        DeriveRevenue(),
        AggregateOrderRevenue(),
    ],
)

q3_pipeline().df.collect().to_native()

l_orderkey,revenue,o_orderdate,o_shippriority
i64,f64,datetime[ns],i32
47714,267010.5894,1995-03-11 00:00:00,0
22276,266351.5562,1995-01-29 00:00:00,0
32965,263768.3414,1995-02-25 00:00:00,0
21956,254541.1285,1995-02-02 00:00:00,0
1637,243512.7981,1995-02-08 00:00:00,0
10916,241320.0814,1995-03-11 00:00:00,0
30497,208566.6969,1995-02-07 00:00:00,0
450,205447.4232,1995-03-05 00:00:00,0
47204,204478.5213,1995-03-13 00:00:00,0


We can sanity-check this against the canonical Q3 — which is just the registry's
`/query/Q3` entry, a `TPCHQuery` instance that runs the vendored narwhals query body
against the same table providers.

In [15]:
canonical = registry["/query/Q3"]().df.to_native()
ours = q3_pipeline().df.collect().to_native()
assert_frame_equal(canonical, ours, check_exact=False)
print("Q3 matches canonical ✓")

Q3 matches canonical ✓


## 8. Confluence: pipelines as inputs to pipelines

Now level up: each side input becomes its own `NarwhalsPipelineModel`. The "main" pipeline
joins them together. This is the **confluence** pattern.

```text
   /table/customer               /table/orders             /table/lineitem
          │                              │                          │
          ▼                              ▼                          ▼
   customer_pipeline             orders_pipeline            lineitem_pipeline
   (filter segment)              (pre-filter date)          (pre-filter + derive)
          │                              │                          │
          └─────── JoinTransform ────────┴────── JoinTransform ─────┘
                                         │
                                         ▼
                              aggregate / sort / head
                                         │
                                         ▼
                                    Q3 result
```

Why bother? Now each side input is independently:

- **Testable** — run `customer_pipeline()` on its own.
- **Configurable** — swap the segment via config without touching the join code.
- **Reusable** — drop the orders pre-filter into another query unchanged.
- **Cacheable** — ccflow's evaluator can memoize each pipeline's result if used in multiple places.

In [16]:
customer_filtered_pipeline = NarwhalsPipelineModel(
    source=registry["/table/customer"],
    transforms=[FilterCustomerSegment(segment="BUILDING")],
)

orders_filtered_pipeline = NarwhalsPipelineModel(
    source=registry["/table/orders"],
    transforms=[FilterOrderDateBefore(cutoff=cutoff)],
)

lineitem_with_revenue_pipeline = NarwhalsPipelineModel(
    source=registry["/table/lineitem"],
    transforms=[FilterShipDateAfter(cutoff=cutoff), DeriveRevenue()],
)

q3_confluence = NarwhalsPipelineModel(
    source=customer_filtered_pipeline,
    transforms=[
        JoinTransform(
            other=orders_filtered_pipeline,
            left_on="c_custkey",
            right_on="o_custkey",
            how="inner",
        ),
        JoinTransform(
            other=lineitem_with_revenue_pipeline,
            left_on="o_orderkey",
            right_on="l_orderkey",
            how="inner",
        ),
        AggregateOrderRevenue(),
    ],
)

ours_conf = q3_confluence().df.collect().to_native()
assert_frame_equal(ours_conf, canonical, check_exact=False)
print("Q3 confluence matches canonical ✓")
ours_conf

Q3 confluence matches canonical ✓


l_orderkey,revenue,o_orderdate,o_shippriority
i64,f64,datetime[ns],i32
47714,267010.5894,1995-03-11 00:00:00,0
22276,266351.5562,1995-01-29 00:00:00,0
32965,263768.3414,1995-02-25 00:00:00,0
21956,254541.1285,1995-02-02 00:00:00,0
1637,243512.7981,1995-02-08 00:00:00,0
10916,241320.0814,1995-03-11 00:00:00,0
30497,208566.6969,1995-02-07 00:00:00,0
450,205447.4232,1995-03-05 00:00:00,0
47204,204478.5213,1995-03-13 00:00:00,0


Notice the structural shift compared to §7:

- Each pre-filter / derivation lives **inside** the pipeline that owns the relevant table,
  rather than being mixed into a single linear list.
- The "main" pipeline now describes only the join-and-aggregate logic.
- Side inputs invoked by `JoinTransform` are not auto-surfaced into the parent's `__deps__`
  (a v1 limitation), but they remain configurable, testable, and serializable.

## 9. `JoinBackTransform`: fork-and-rejoin patterns

Sometimes a transform needs to compute a per-group summary and **enrich** the original rows
with it. Two tools cover the spectrum:

- **`with_columns(expr.over(window))`** — the right answer when the summary fits a window
  function (same row count as input). Prefer this when it applies.
- **`JoinBackTransform`** — runs an inner transform on the input to produce a summary
  with potentially different shape, then joins it back. Useful when the summary aggregates
  across groups, or projects to a different column set.

In [17]:
toy_lineitem = nw.from_native(
    pl.LazyFrame(
        {
            "l_orderkey": [1, 1, 2, 2, 3],
            "l_extendedprice": [100.0, 50.0, 200.0, 80.0, 30.0],
            "l_discount": [0.0, 0.1, 0.0, 0.05, 0.0],
        }
    )
)


class TotalRevenuePerOrder(NarwhalsFrameTransform):
    """Inner transform: compute total revenue per order key."""

    def __call__(self, df):
        return (
            df.with_columns(revenue=nw.col("l_extendedprice") * (1 - nw.col("l_discount")))
            .group_by("l_orderkey")
            .agg(nw.sum("revenue").alias("order_total_revenue"))
        )


enrich_with_order_total = JoinBackTransform(inner=TotalRevenuePerOrder(), on="l_orderkey", how="left")

toy_lineitem.pipe(enrich_with_order_total).collect().to_native()

l_orderkey,l_extendedprice,l_discount,order_total_revenue
i64,f64,f64,f64
1,100.0,0.0,145.0
1,50.0,0.1,145.0
2,200.0,0.0,276.0
2,80.0,0.05,276.0
3,30.0,0.0,30.0


For per-row deltas (e.g. "this line item's revenue minus its order's average"), the
window-function form is cleaner:

```python
df.with_columns(
    delta=(nw.col("l_extendedprice") * (1 - nw.col("l_discount")))
          - (nw.col("l_extendedprice") * (1 - nw.col("l_discount"))).mean().over("l_orderkey")
)
```

`JoinBackTransform` earns its place when the summary's shape genuinely differs from the
input — e.g. multi-column projections, or aggregations across multiple keys.

## 10. When *not* to use `NarwhalsPipelineModel`

The linear `.pipe()` contract is the design's strength — but it's also a constraint.
Some queries don't fit:

- **True DAGs.** TPC-H Q15 (CTE referenced from two places) and Q21 (correlated EXISTS)
  share intermediate frames across multiple downstream consumers. Forcing them through
  a single `.pipe()` chain duplicates work.
- **Scalar subqueries.** Q4, Q11, Q17, Q22 compute a scalar (or a tiny frame) from one
  query and use it as a parameter in another. The "pipeline" view is the wrong abstraction.
- **Multi-output stages.** Anything that wants to fork into multiple labeled outputs
  (e.g. `frame -> {"by_segment": ..., "by_region": ...}`) breaks the `LazyFrame -> LazyFrame`
  contract.

The right move for these is to step **up** a layer: split the query into multiple
`CallableModel`s, compose them at the graph level, and let ccflow's evaluator handle
parallelization and caching of shared upstream computations. The pipeline abstractions in
this notebook live *inside* each node of that graph; they're not a replacement for it.

## 11. Summary

Three layers, each opt-in:

1. **`NarwhalsFrameTransform`** — pydantic + callable + `.pipe()`-compatible. Use anywhere
   narwhals is used; no ccflow runtime required by callers. Subclassing is a *convention*,
   not a requirement — see §4 for the plain-function / plain-`BaseModel` / `NarwhalsFrameTransform`
   tradeoffs.
2. **`SequenceTransform`** — bundle. Nestable, JSON-roundtrippable when transforms are
   model-shaped.
3. **`NarwhalsPipelineModel`** — full ccflow integration: sources, contexts, dependency
   declarations, evaluator, caching, Hydra config.

Plus two ready-made transforms:

- **`JoinTransform`** — multi-source enrichment with another `CallableModel`.
- **`JoinBackTransform`** — fork-and-rejoin when window functions don't fit.

When pipelines compose pipelines, you get **confluence** — small, testable units glued
together at the join boundary, each independently configurable and (in future) cacheable.

When the query exceeds linear `.pipe()`, step up to multi-`CallableModel` graphs.